# exp034_trt_cardiovascular

**Track 4 — Endocrine & TRT**

**Experiment:** `exp034`
**Description:** Simple hazard ratio model based on Sharma 2015.
    If T is normalized (above threshold): HR = hr_normalized
    If T remains below threshold: HR = hr_untreated
    Linear interpolation in between.

**Source:** `control/endocrine/exp034_trt_cardiovascular.py`

## Paper Linkage

See `specs/PAPER_REVIEW_QUEUE.md` for the full paper → experiment mapping.
This notebook is the `.ipynb` pair of the `.py` control script for `exp034`.


In [ ]:
"""
healthSpring Exp034 — TRT Cardiovascular Response

Models cardiovascular biomarker changes under TRT:
  - LDL cholesterol reduction (Saad 2016)
  - HDL cholesterol increase (Saad 2016)
  - CRP reduction (inflammation marker, Saad 2016)
  - Systolic/diastolic blood pressure reduction (Saad 2016)
  - Interrupted vs continuous comparison

Reference: Saad 2016 (continuous n=115, interrupted n=147)
           Sharma 2015 (VA, n=83,010): HR=0.44 for MI with T normalization
           Muraleedharan 2013: all-cause mortality
           Mok Ch.6

Provenance:
  Baseline date:   2026-03-08
  Command:         python3 control/endocrine/exp034_trt_cardiovascular.py
  Environment:     Python 3.10+, NumPy
"""

import json
import os
import sys

import numpy as np

SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))

# Saad 2016 baseline and endpoint values (digitized from published figures)
# All continuous group values

# LDL (mg/dL)
LDL_BASELINE = 165.0
LDL_ENDPOINT = 130.0  # after ~5 years continuous TRT

# HDL (mg/dL)
HDL_BASELINE = 38.0
HDL_ENDPOINT = 55.0

# CRP (mg/dL)
CRP_BASELINE = 1.40
CRP_ENDPOINT = 0.90

# Systolic BP (mmHg)
SBP_BASELINE = 135.0
SBP_ENDPOINT = 123.0

# Diastolic BP (mmHg)
DBP_BASELINE = 82.0
DBP_ENDPOINT = 76.0

MONTHS = np.arange(0, 61, dtype=float)
TAU_MONTHS = 12.0  # characteristic time for CV improvements


def biomarker_trajectory(months, baseline_val, endpoint_val, tau):
    """Exponential approach to new setpoint."""
    delta = endpoint_val - baseline_val
    return baseline_val + delta * (1.0 - np.exp(-months / tau))


def hazard_ratio_model(t_level, threshold=300.0, hr_normalized=0.44, hr_untreated=1.0):
    """
    Simple hazard ratio model based on Sharma 2015.
    If T is normalized (above threshold): HR = hr_normalized
    If T remains below threshold: HR = hr_untreated
    Linear interpolation in between.
    """
    if t_level >= threshold:
        return hr_normalized
    ratio = t_level / threshold
    return hr_untreated - (hr_untreated - hr_normalized) * ratio


def main():
    total_passed = 0
    total_failed = 0
    baseline = {}

    print("=" * 72)
    print("healthSpring Exp034: TRT Cardiovascular Response")
    print("=" * 72)

    ldl = biomarker_trajectory(MONTHS, LDL_BASELINE, LDL_ENDPOINT, TAU_MONTHS)
    hdl = biomarker_trajectory(MONTHS, HDL_BASELINE, HDL_ENDPOINT, TAU_MONTHS)
    crp = biomarker_trajectory(MONTHS, CRP_BASELINE, CRP_ENDPOINT, TAU_MONTHS)
    sbp = biomarker_trajectory(MONTHS, SBP_BASELINE, SBP_ENDPOINT, TAU_MONTHS)
    dbp = biomarker_trajectory(MONTHS, DBP_BASELINE, DBP_ENDPOINT, TAU_MONTHS)


In [ ]:
# --- Check 1: All baselines correct at t=0 ---
    print("\n--- Check 1: Baseline values at t=0 ---")
    ok = (abs(ldl[0] - LDL_BASELINE) < 1e-10 and
          abs(hdl[0] - HDL_BASELINE) < 1e-10 and
          abs(crp[0] - CRP_BASELINE) < 1e-10 and
          abs(sbp[0] - SBP_BASELINE) < 1e-10 and
          abs(dbp[0] - DBP_BASELINE) < 1e-10)
    baseline["baselines_correct"] = ok
    if ok:
        print(f"  [PASS] All baselines match at t=0")
        total_passed += 1
    else:
        print(f"  [FAIL] Baseline mismatch")
        total_failed += 1


In [ ]:
# --- Check 2: LDL decreases ---
    print("\n--- Check 2: LDL decreases with TRT ---")
    ldl_60 = float(ldl[-1])
    baseline["ldl_60mo"] = ldl_60
    if ldl_60 < LDL_BASELINE:
        print(f"  [PASS] LDL: {LDL_BASELINE:.0f} → {ldl_60:.1f} mg/dL")
        total_passed += 1
    else:
        print(f"  [FAIL] LDL did not decrease: {ldl_60:.1f}")
        total_failed += 1


In [ ]:
# --- Check 3: HDL increases ---
    print("\n--- Check 3: HDL increases with TRT ---")
    hdl_60 = float(hdl[-1])
    baseline["hdl_60mo"] = hdl_60
    if hdl_60 > HDL_BASELINE:
        print(f"  [PASS] HDL: {HDL_BASELINE:.0f} → {hdl_60:.1f} mg/dL")
        total_passed += 1
    else:
        print(f"  [FAIL] HDL did not increase: {hdl_60:.1f}")
        total_failed += 1


In [ ]:
# --- Check 4: CRP decreases (anti-inflammatory) ---
    print("\n--- Check 4: CRP decreases (anti-inflammatory) ---")
    crp_60 = float(crp[-1])
    baseline["crp_60mo"] = crp_60
    if crp_60 < CRP_BASELINE:
        print(f"  [PASS] CRP: {CRP_BASELINE:.2f} → {crp_60:.2f} mg/dL")
        total_passed += 1
    else:
        print(f"  [FAIL] CRP did not decrease: {crp_60:.2f}")
        total_failed += 1


In [ ]:
# --- Check 5: Blood pressure normalizes ---
    print("\n--- Check 5: Blood pressure decreases ---")
    sbp_60 = float(sbp[-1])
    dbp_60 = float(dbp[-1])
    baseline["sbp_60mo"] = sbp_60
    baseline["dbp_60mo"] = dbp_60
    if sbp_60 < SBP_BASELINE and dbp_60 < DBP_BASELINE:
        print(f"  [PASS] SBP: {SBP_BASELINE:.0f} → {sbp_60:.1f}, DBP: {DBP_BASELINE:.0f} → {dbp_60:.1f} mmHg")
        total_passed += 1
    else:
        print(f"  [FAIL] BP: SBP={sbp_60:.1f}, DBP={dbp_60:.1f}")
        total_failed += 1


In [ ]:
# --- Check 6: SBP enters normal range (<120 was normal, <130 acceptable) ---
    print("\n--- Check 6: SBP approaches normal range ---")
    baseline["sbp_in_range"] = sbp_60 < 130.0
    if sbp_60 < 130.0:
        print(f"  [PASS] SBP = {sbp_60:.1f} mmHg (< 130 threshold)")
        total_passed += 1
    else:
        print(f"  [FAIL] SBP = {sbp_60:.1f} mmHg (still ≥ 130)")
        total_failed += 1


In [ ]:
# --- Check 7: Improvements are front-loaded (exponential approach) ---
    print("\n--- Check 7: Front-loaded improvements ---")
    ldl_12 = float(ldl[12])
    ldl_delta_12 = (LDL_BASELINE - ldl_12) / (LDL_BASELINE - ldl_60) if abs(LDL_BASELINE - ldl_60) > 0 else 0
    baseline["ldl_frac_improvement_12mo"] = ldl_delta_12
    if ldl_delta_12 > 0.55:
        print(f"  [PASS] {ldl_delta_12:.1%} of LDL improvement by 12 months (exponential)")
        total_passed += 1
    else:
        print(f"  [FAIL] Only {ldl_delta_12:.1%} by 12 months")
        total_failed += 1


In [ ]:
# --- Check 8: Hazard ratio model ---
    print("\n--- Check 8: Hazard ratio with T normalization ---")
    hr_low = hazard_ratio_model(200.0)
    hr_norm = hazard_ratio_model(600.0)
    hr_mid = hazard_ratio_model(300.0)
    baseline["hr_at_200"] = hr_low
    baseline["hr_at_300"] = hr_mid
    baseline["hr_at_600"] = hr_norm
    if hr_norm <= hr_mid < hr_low:
        print(f"  [PASS] HR: T=200 → {hr_low:.2f}, T=300 → {hr_mid:.2f}, T=600 → {hr_norm:.2f}")
        total_passed += 1
    else:
        print(f"  [FAIL] HR ordering violated")
        total_failed += 1


In [ ]:
# --- Check 9: HR at normalization matches Sharma 2015 ---
    print("\n--- Check 9: HR at normalization = 0.44 (Sharma 2015) ---")
    if abs(hr_norm - 0.44) < 1e-10:
        print(f"  [PASS] HR(normalized) = {hr_norm:.2f} = 0.44")
        total_passed += 1
    else:
        print(f"  [FAIL] HR(normalized) = {hr_norm:.2f}")
        total_failed += 1


In [ ]:
# --- Check 10: All biomarker trajectories smooth ---
    print("\n--- Check 10: All trajectories smooth (no reversals) ---")
    ldl_mono = all(ldl[i] >= ldl[i + 1] - 1e-12 for i in range(len(ldl) - 1))
    hdl_mono = all(hdl[i] <= hdl[i + 1] + 1e-12 for i in range(len(hdl) - 1))
    crp_mono = all(crp[i] >= crp[i + 1] - 1e-12 for i in range(len(crp) - 1))
    sbp_mono = all(sbp[i] >= sbp[i + 1] - 1e-12 for i in range(len(sbp) - 1))
    baseline["all_smooth"] = ldl_mono and hdl_mono and crp_mono and sbp_mono
    if ldl_mono and hdl_mono and crp_mono and sbp_mono:
        print(f"  [PASS] All trajectories monotonic (smooth approach to setpoint)")
        total_passed += 1
    else:
        print(f"  [FAIL] Non-monotonic trajectory detected")
        total_failed += 1


In [ ]:
# --- Write baseline JSON ---
    baseline_path = os.path.join(SCRIPT_DIR, "exp034_baseline.json")
    baseline_out = {
        "_source": "healthSpring Exp034: TRT Cardiovascular Response",
        "_method": "Exponential approach to setpoint + Sharma hazard ratio",
        "params": {
            "LDL": {"baseline": LDL_BASELINE, "endpoint": LDL_ENDPOINT},
            "HDL": {"baseline": HDL_BASELINE, "endpoint": HDL_ENDPOINT},
            "CRP": {"baseline": CRP_BASELINE, "endpoint": CRP_ENDPOINT},
            "SBP": {"baseline": SBP_BASELINE, "endpoint": SBP_ENDPOINT},
            "DBP": {"baseline": DBP_BASELINE, "endpoint": DBP_ENDPOINT},
            "tau_months": TAU_MONTHS,
        },
        **baseline,
        "_provenance": {
            "date": "2026-03-08",
            "python": sys.version,
            "numpy": np.__version__,
        },
    }
    with open(baseline_path, "w") as f:
        json.dump(baseline_out, f, indent=2, default=str)
    print(f"\nBaseline written to {baseline_path}")

    total = total_passed + total_failed
    print(f"\n{'=' * 72}")
    print(f"TOTAL: {total_passed}/{total} PASS, {total_failed}/{total} FAIL")
    print(f"{'=' * 72}")

    return 0 if total_failed == 0 else 1


if __name__ == "__main__":
    sys.exit(main())
